# RAG with MiniMax

This notebook demonstrates how to use **[MiniMax](https://platform.minimaxi.com/)** as an alternative LLM and embedding provider in the bRAG-langchain RAG pipeline.

MiniMax exposes an **OpenAI-compatible** chat API, so LangChain's `ChatOpenAI` works out of the box — just point it at `https://api.minimax.io/v1`.

For embeddings, MiniMax's `embo-01` model uses a proprietary format (`texts` + `type` fields), so we provide a dedicated `MiniMaxEmbeddings` wrapper in `utils/minimax_embeddings.py`.

**What you'll learn:**
1. How to configure MiniMax as your LLM provider
2. How to use MiniMax embeddings with the native API
3. How to build a complete RAG pipeline with MiniMax

**Models used:**
- Chat: `MiniMax-M2.7` (up to 1M context window)
- Embeddings: `embo-01` (1536 dimensions)

## 1. Environment Setup

In [ ]:
import os
import sys

# Add the project root to the path so we can import utils
sys.path.insert(0, os.path.join(os.getcwd(), ".."))

from dotenv import load_dotenv

load_dotenv()

# Verify the MiniMax API key is set
assert os.getenv("MINIMAX_API_KEY"), "Please set MINIMAX_API_KEY in your .env file"
print("MiniMax API key loaded.")

## 2. Using the Provider Factory

The `utils/llm_provider.py` module provides a simple factory to create LLM and embedding instances for any supported provider.

In [ ]:
from utils.llm_provider import get_chat_model, get_embeddings, PROVIDER_DEFAULTS

# Show available providers
for name, cfg in PROVIDER_DEFAULTS.items():
    print(f"  {name}: chat={cfg['chat_model']}, embedding={cfg['embedding_model']}")

In [ ]:
# Create a MiniMax chat model
llm = get_chat_model(provider="minimax")

# Quick test
response = llm.invoke("What is Retrieval-Augmented Generation in one sentence?")
print(response.content)

## 3. MiniMax Embeddings

MiniMax's `embo-01` embedding model uses a proprietary API format. The `MiniMaxEmbeddings` class wraps this into LangChain's `Embeddings` interface.

In [ ]:
from utils.minimax_embeddings import MiniMaxEmbeddings

embeddings = MiniMaxEmbeddings()

# Embed a single query
query_vec = embeddings.embed_query("What is RAG?")
print(f"Query vector dimension: {len(query_vec)}")
print(f"First 5 values: {query_vec[:5]}")

In [ ]:
# Embed multiple documents
docs_text = [
    "RAG combines retrieval with generation.",
    "LangChain is a framework for LLM applications.",
    "MiniMax provides large language models via API.",
]
doc_vecs = embeddings.embed_documents(docs_text)
print(f"Embedded {len(doc_vecs)} documents, each with {len(doc_vecs[0])} dimensions.")

## 4. Complete RAG Pipeline with MiniMax

Below we build a full RAG pipeline using:
- **MiniMax M2.7** for chat generation
- **MiniMax embo-01** for document embeddings
- **ChromaDB** as the local vector store (no Pinecone required)

In [ ]:
from langchain_community.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import Chroma
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain.prompts import ChatPromptTemplate

# Load the sample PDF
pdf_path = os.path.join("..", "test", "langchain_turing.pdf")
if not os.path.exists(pdf_path):
    pdf_path = "langchain_turing.pdf"  # fallback for notebooks/ dir

loader = PyPDFLoader(pdf_path)
docs = loader.load()
print(f"Loaded {len(docs)} pages from PDF.")

In [ ]:
# Split documents into chunks
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
splits = text_splitter.split_documents(docs)
print(f"Split into {len(splits)} chunks.")

In [ ]:
# Create a local ChromaDB vector store with MiniMax embeddings
minimax_embeddings = MiniMaxEmbeddings()

vectorstore = Chroma.from_documents(
    documents=splits,
    embedding=minimax_embeddings,
    collection_name="brag_minimax_demo",
)

retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
print("Vector store created with MiniMax embeddings.")

In [ ]:
# Build the RAG chain
template = """Answer the question based only on the following context:
{context}

Question: {question}
"""
prompt = ChatPromptTemplate.from_template(template)

# MiniMax M2.7 as the LLM
minimax_llm = get_chat_model(provider="minimax", model="MiniMax-M2.7", temperature=0.1)


def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)


rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | minimax_llm
    | StrOutputParser()
)
print("RAG chain ready.")

In [ ]:
# Ask a question!
from pprint import pprint

answer = rag_chain.invoke("What is this document about?")
pprint(answer)

In [ ]:
# Try another question
answer2 = rag_chain.invoke("What are the key components of LangChain?")
pprint(answer2)

## 5. Switching Providers via Environment Variables

You can switch between providers without changing any code. Just set:

```env
# In your .env file
LLM_PROVIDER=minimax       # or "openai"
MINIMAX_API_KEY=your-key
```

Then `get_chat_model()` and `get_embeddings()` will automatically use the configured provider.

In [ ]:
# Example: use environment variable to switch provider
os.environ["LLM_PROVIDER"] = "minimax"

# get_chat_model() now returns a MiniMax-backed ChatOpenAI
auto_llm = get_chat_model()
response = auto_llm.invoke("Say hello in three languages.")
print(response.content)

## Summary

In this notebook you learned how to:
- Use `get_chat_model(provider="minimax")` to create a MiniMax LLM
- Use `MiniMaxEmbeddings` for document/query embeddings with `embo-01`
- Build a full RAG pipeline with MiniMax + ChromaDB
- Switch providers via environment variables

For more about MiniMax models, visit [platform.minimaxi.com](https://platform.minimaxi.com/).